# 06. Análisis y Modelos de Negocio

Este notebook utiliza el dataset consolidado para responder a las 5 preguntas estratégicas del Sunset Hospitality Group mediante estadística descriptiva y aprendizaje automático (Machine Learning).

**Preguntas a responder:**
1. Tasa real de duplicidad.
2. Factores que influyen en la cancelación (Random Forest).
3. Clasificación de reservas de alto/bajo valor.
4. Perfiles de huéspedes recurrentes (K-Means).
5. Tasa real de cancelaciones.

In [8]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Configuración de rutas
PROCESSED_PATH = '../data/processed/'
RESULTS_PATH = '../data/results/'

# Carga de datos
df = pd.read_csv(os.path.join(PROCESSED_PATH, 'consolidated_bookings.csv'))
df_unified = pd.read_csv(os.path.join(PROCESSED_PATH, 'unified_dataset.csv')) # Para calcular duplicidad

## Pregunta 1: Tasa Real de Duplicidad
Calculamos qué porcentaje del dataset unificado representaban registros duplicados fusionados.

In [9]:
total_unificado = len(df_unified)
total_consolidado = len(df)
duplicados = total_unificado - total_consolidado
tasa_duplicidad = (duplicados / total_unificado) * 100

print(f"Tasa de Duplicidad Real: {tasa_duplicidad:.2f}%")
print(f"Registros fusionados: {duplicados}")

Tasa de Duplicidad Real: 48.13%
Registros fusionados: 114924


## Pregunta 2: Factores de Cancelación
Entrenamos un Random Forest para identificar las variables que más impactan en `is_canceled`.

In [10]:
# Preparación de datos (Encoding de categóricas)
le = LabelEncoder()
df_model = df.copy()

categorical_cols = ['hotel', 'meal', 'market_segment', 'deposit_type', 'customer_type']
for col in categorical_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

# Selección de features numéricas y categóricas encodeadas
features = ['lead_time', 'total_special_requests', 'average_daily_rate', 'booking_changes', 
            'deposit_type', 'market_segment', 'previous_cancellations']

X = df_model[features]
y = df_model['is_canceled']

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

# Importancia de variables
importances = pd.DataFrame({
    'feature': features,
    'importance': rf.feature_importances_
}).sort_values(by='importance', ascending=False)

print("Variables más influyentes en cancelaciones:")
print(importances)

Variables más influyentes en cancelaciones:
                  feature  importance
2      average_daily_rate    0.330337
0               lead_time    0.296580
4            deposit_type    0.169920
5          market_segment    0.066324
1  total_special_requests    0.062427
6  previous_cancellations    0.046686
3         booking_changes    0.027727


## Pregunta 3: Clasificación Alto/Bajo Valor
Definimos 'Alto Valor' como reservas por encima del percentil 75 de la tarifa diaria (`average_daily_rate`).

In [11]:
umbral_valor = df['average_daily_rate'].quantile(0.75)
df['is_high_value'] = (df['average_daily_rate'] > umbral_valor).astype(int)

conteo_valor = df['is_high_value'].value_counts(normalize=True) * 100
print(f"Distribución de valor (Umbral P75 = {umbral_valor:.2f}):")
print(f"Bajo Valor: {conteo_valor[0]:.2f}% | Alto Valor: {conteo_valor[1]:.2f}%")

Distribución de valor (Umbral P75 = 126.00):
Bajo Valor: 75.59% | Alto Valor: 24.41%


## Pregunta 4: Perfiles de Huéspedes Recurrentes (Clustering)
Analizamos el comportamiento de los clientes que ya se han hospedado previamente.

In [12]:
df_recurrent = df[df['is_repeated_guest'] == 1].copy()

cluster_features = ['lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 'total_special_requests']
X_clust = StandardScaler().fit_transform(df_recurrent[cluster_features])

kmeans = KMeans(n_clusters=3, random_state=42)
df_recurrent['cluster'] = kmeans.fit_predict(X_clust)

print("Perfiles de huéspedes recurrentes (Promedios por Cluster):")
print(df_recurrent.groupby('cluster')[cluster_features].mean())

Perfiles de huéspedes recurrentes (Promedios por Cluster):
          lead_time  stays_in_weekend_nights  stays_in_week_nights  \
cluster                                                              
0         10.439191                 0.443775              1.058697   
1         49.762214                 0.196183              1.508270   
2        123.107143                 2.767857              6.672619   

         total_special_requests  
cluster                          
0                      0.081798  
1                      1.320611  
2                      1.398810  


## Pregunta 5: Tasa Real de Cancelaciones
Cálculo final tras la unificación de estatus.

In [13]:
tasa_cancelacion_final = df['is_canceled'].mean() * 100
print(f"Tasa real de cancelaciones (post-consolidación): {tasa_cancelacion_final:.2f}%")

Tasa real de cancelaciones (post-consolidación): 37.02%


## Guardar Resultados
Exportamos las métricas clave para el reporte final.

In [14]:
results = {
    'metric': ['tasa_duplicidad', 'tasa_cancelacion', 'umbral_alto_valor'],
    'value': [tasa_duplicidad, tasa_cancelacion_final, umbral_valor]
}

pd.DataFrame(results).to_csv(os.path.join(RESULTS_PATH, 'model_results.csv'), index=False)
print("[OK] Resultados del análisis guardados en data/results/model_results.csv")

[OK] Resultados del análisis guardados en data/results/model_results.csv
